In [1]:
from pathlib import Path
import sys
import os
import clingo
project_root = "."


In [2]:
from typing import Any

INSTANCE = "tree"

ASPGARP_FILES = [
    f"{project_root}/single/sim.lp",
    f"{project_root}/single/cnst.lp",
    f"{project_root}/adapter.lp",
]

REFERENCE_MODEL = [
    f"{project_root}/models/{INSTANCE}/db.lp",
    f"{project_root}/models/{INSTANCE}/ref.lp",
]

BASIC_MODEL = [
    f"{project_root}/models/{INSTANCE}/db.lp"
]

META_FILES = [
    f"{project_root}/meta.lp",
]

OUTPUT = f"{project_root}/out/{INSTANCE}"

#create output directory if it doesn't exist
os.makedirs(OUTPUT, exist_ok=True)

In [3]:
import clingo


def states_from_reference_model(
    RMODEL: list[str], ASPGARPFILES: list[str]
) -> list[dict[str, tuple[str, str]]]:
    states: list[dict[str, tuple[str, str]]] = []

    ctl = clingo.Control(["0", "--project", "--warn=none"])
    for path in RMODEL + ASPGARPFILES:
        ctl.load(path)

    ctl.add("base", [], "#show obs/1.")
    ctl.add("base", [], ":- violation.")
    ctl.ground([("base", [])])

    with ctl.solve(yield_=True) as handle:
        for model in handle:
            state: dict[str, tuple[str, str]] = {}
            for atom in model.symbols(shown=True):
                if atom.name == "obs" and len(atom.arguments) == 1:
                    varname, value, direction = atom.arguments[0].arguments
                    state[varname.name] = (value.name, direction.name)
            states.append(state)

    return states


def states_to_description(
    states: list[dict[str, tuple[str, str]]]
) -> list[str]:
    descriptions: list[str] = []

    for i, state in enumerate(states):
        parts = [f"holds(SID,{varname},{value},{direction})"
                 for varname, (value, direction) in state.items()]
        description = f"state(SID,{i}) :- " + ", ".join(parts) + "."
        descriptions.append(description)

    return descriptions

def states_to_labels(states: list[dict[str, tuple[str, str]]]) -> list[str]:
    labels: list[str] = []

    for i, state in enumerate(states):
        parts = [f"{varname}=({value},{direction})"
                 for varname, (value, direction) in state.items()]
        label = f"statelabel({i},\"" + ",\\n ".join(parts) + "\")."
        labels.append(label)

    return labels


In [4]:
# Reuse the above and find all the transitions between them

def transitions_from_reference_model(RMODEL, ASPGARPFILES, state_descriptions) -> list[str]:
    transitions = []
    ctl = clingo.Control(["0", "--project", "--warn=none"])
    for path in REFERENCE_MODEL + ASPGARP_FILES:
        ctl.load(path)
    ctl.add("base", [],  "\n".join(state_descriptions))
    ctl.add("base", [], ":- violation.")
    ctl.add("base", [], "#show state/2.")
    ctl.ground([("base", [])])

    for i, m in enumerate(ctl.solve(yield_=True)):
        from_state = None
        to_state = None
        for atom in m.symbols(shown=True):
            if atom.name == "state" and len(atom.arguments) == 2:
                if atom.arguments[0] == clingo.Number(0): 
                    from_state = atom
                else:
                    to_state = atom
        if from_state and to_state:
            transitions.append((from_state.arguments[1], to_state.arguments[1]))
    return transitions

# Step 0: Generate Positive Examples

In [5]:

states = states_from_reference_model(REFERENCE_MODEL, ASPGARP_FILES)
state_descriptions = states_to_description(states)
state_labels = states_to_labels(states)
transitions = transitions_from_reference_model(REFERENCE_MODEL, ASPGARP_FILES, state_descriptions)
transition_descriptions = [f"edge(({from_state},{to_state}))." for from_state, to_state in transitions]

#write the state and transition descriptions to files
with open(f"{OUTPUT}/transitions.lp", "w") as f:
    f.write("\n".join(transition_descriptions))
    f.write("\n".join(state_labels))

with open(f"{OUTPUT}/states.lp", "w") as f:
    f.write("\n".join(state_descriptions))
    

!clingo --project --warn=none --outf=2 {OUTPUT}/transitions.lp  | clingraph --out=render --format=svg --viz-encoding=viz.lp

-> Image for graph default, saved in: out/0/default.svg


# Step 1: 

## Pruning



In [6]:
states

def state_to_assumptions(state) -> list[tuple[clingo.Symbol, bool]]:
    assumptions: list[tuple[clingo.Symbol, bool]] = []
    for varname, (value, direction) in state.items():
        atom = clingo.Function(
            "holds",
            [
                clingo.Function(varname),
                clingo.Function(value),
                clingo.Function(direction),
            ],
        )
        wrapper = clingo.Function("obs", [atom])
        assumptions.append((wrapper, True))
    return assumptions
print(state_to_assumptions(states[0]))

[(Function('obs', [Function('holds', [Function('growth', [], True), Function('none', [], True), Function('decrease', [], True)], True)], True), True), (Function('obs', [Function('holds', [Function('shade', [], True), Function('small', [], True), Function('decrease', [], True)], True)], True), True), (Function('obs', [Function('holds', [Function('size', [], True), Function('small', [], True), Function('decrease', [], True)], True)], True), True)]


In [7]:
def find_hypotheses(
    pos_states,
    aspqsim_files: list[str],
    model_files: list[str],
    meta_files: list[str],
    *,
    verbose: bool = True,
    options = [],
    hypothesis_constraints = []
) -> list[list[tuple[clingo.Symbol, bool]]]:
    """
    Search hypotheses for positive examples and accumulate nogoods
    to prune previously accepted solutions.
    """
    files = aspqsim_files + model_files + meta_files

    ctl = clingo.Control(["0", "--project", "--warn=none"] + options)
    for path in files:
        ctl.load(path)

    ctl.add("base", [], "\n".join(hypothesis_constraints))
    ctl.add("base", [], "#show abduced/1.")
    ctl.ground([("base", [])])

    violation = clingo.parse_term("violation")
    hypothesis_constraints: list[list[tuple[clingo.Symbol, bool]]] = []

    for key, pos_example in enumerate(pos_states):
        if verbose:
            print(f"Example ID: {key}")

        base_assumptions = state_to_assumptions(pos_example)

        if verbose:
            for atom, _ in base_assumptions:
                print(f"Adding assumption: {atom}")

        solve_assumptions =  base_assumptions + [(violation, True)]
        found_valid_model = False
        added_existing_nogoods = False

        with ctl.solve(yield_=True, assumptions=solve_assumptions) as handle:
            counter = 0
            for model in handle:
                if verbose:
                    print("Model found, checking for violation...")

                if not added_existing_nogoods:
                    for nogood in hypothesis_constraints:
                        model.context.add_nogood(nogood)
                    added_existing_nogoods = True

                atoms = model.symbols(shown=True)

                # Skip models already ruled out by an existing nogood.
                if any(all(atom in atoms for atom, _ in nogood) for nogood in hypothesis_constraints):
                    if verbose:
                        print("Fluke detected, skipping this model.")
                    continue

                found_valid_model = True

                if verbose:
                    print(atoms)

                nogood = [(atom, True) for atom in atoms]
                hypothesis_constraints.append(nogood)
                model.context.add_nogood(nogood)

        if not found_valid_model and verbose:
            print(
                "No model found for this example, even without the nogood constraints. "
            )
    print(counter)
    return hypothesis_constraints

In [8]:
hypothesis_constraints = find_hypotheses(states, ASPGARP_FILES, BASIC_MODEL, META_FILES, verbose=True)

constraints = []

for nogood in hypothesis_constraints:
    constraints.append(":- " + ", ".join(f"{atom}".replace("rl(1)","RULE1").replace("rl(2)","RULE2") for atom, val in nogood) + ".")

print(len(constraints), "constraints generated from nogoods.")

constraints
    

Example ID: 0
Adding assumption: obs(holds(growth,none,decrease))
Adding assumption: obs(holds(shade,small,decrease))
Adding assumption: obs(holds(size,small,decrease))
Model found, checking for violation...
[abduced(proportionality(-1,size,shade))]
Model found, checking for violation...
[abduced(proportionality(-1,size,growth))]
Model found, checking for violation...
[abduced(proportionality(-1,growth,shade))]
Model found, checking for violation...
[abduced(proportionality(-1,shade,growth))]
Model found, checking for violation...
[abduced(proportionality(-1,shade,size))]
Model found, checking for violation...
[abduced(proportionality(-1,growth,size))]
Example ID: 1
Adding assumption: obs(holds(growth,none,neutral))
Adding assumption: obs(holds(shade,small,neutral))
Adding assumption: obs(holds(size,small,neutral))
No model found for this example, even without the nogood constraints. 
Example ID: 2
Adding assumption: obs(holds(growth,none,increase))
Adding assumption: obs(holds(shade,l

[':- abduced(proportionality(-1,size,shade)).',
 ':- abduced(proportionality(-1,size,growth)).',
 ':- abduced(proportionality(-1,growth,shade)).',
 ':- abduced(proportionality(-1,shade,growth)).',
 ':- abduced(proportionality(-1,shade,size)).',
 ':- abduced(proportionality(-1,growth,size)).',
 ':- abduced(influence(-1,size,shade)).',
 ':- abduced(influence(-1,shade,growth)).',
 ':- abduced(correspondence(size,growth)).',
 ':- abduced(correspondence(growth,shade)).',
 ':- abduced(correspondence(shade,growth)).',
 ':- abduced(correspondence(growth,size)).',
 ':- abduced(influence(-1,shade,size)).',
 ':- abduced(influence(-1,size,growth)).',
 ':- abduced(influence(-1,growth,size)).',
 ':- abduced(influence(-1,growth,shade)).']

In [9]:
transitions

[(Number(0), Number(0)),
 (Number(7), Number(7)),
 (Number(1), Number(1)),
 (Number(7), Number(4)),
 (Number(3), Number(3)),
 (Number(4), Number(4)),
 (Number(4), Number(3)),
 (Number(2), Number(3)),
 (Number(2), Number(2)),
 (Number(5), Number(4)),
 (Number(5), Number(5)),
 (Number(5), Number(2)),
 (Number(5), Number(3)),
 (Number(6), Number(4)),
 (Number(6), Number(5)),
 (Number(6), Number(7)),
 (Number(6), Number(6))]

# Step 2:

## Justification Assignments

Find all Rules for every *single* Variables (if needed) i.e. if there is observed change, this change needs to be justified. 

For each transition, for every variable, do you need a causal dependency to justify the change if so list them. Otherwise skip.

In [10]:
import clingo
from typing import Any, Iterable, Mapping


def make_clingo_control(
    aspqsim_files: Iterable[str],
    meta_files: Iterable[str],
    model_files: Iterable[str],
    constraints: Iterable[str],
    *,
    options: Iterable[str] = [],
    parallel_mode: int = 4,
) -> clingo.Control:
    """Create and ground a clingo Control object."""
    ctl = clingo.Control([
        "0",
        "--project",
        "--warn=none",
        *options,
    ])

    for path in [*aspqsim_files, *meta_files, *model_files]:
        ctl.load(path)

    ctl.add("base", [], "\n".join(constraints))
    ctl.add("base", [], "#show abduced/1.")
    ctl.ground([("base", [])])
    return ctl


def build_jusification_assignments(
    transitions,
    ctl: clingo.Control,
    *,
    require_success: bool = True,
    verbose: bool = False,
) -> list[str]:
    """Solve each counterexample and return assignment strings."""
    violation = clingo.Function("violation")
    assignments: list[str] = []
    term_cache: dict[str, clingo.Symbol] = {}

    for key, transition in enumerate(transitions):
        if verbose:
            print(f"Positive Example (Transition) ID: {key}")

        assumptions = [
            (clingo.Function("transition", [transition[0], transition[1]]), True)
        ]

        if require_success:
            assumptions.append((violation, False))

        with ctl.solve(yield_=True, assumptions=assumptions) as handle:
            for model in handle:
                shown = sorted(model.symbols(shown=True), key=str)
                rule = "(" + ",".join(map(str, shown)) + ")"
                assignment = f"in({key}, {rule})"
                assignments.append(assignment)

                if verbose:
                    print(assignment)

    return assignments

In [11]:
options = ["--const", "c = 0"]

ctl = make_clingo_control(ASPGARP_FILES + [f"{OUTPUT}/transitions.lp", f"{OUTPUT}/states.lp", f"{project_root}/transitionobs.lp"], META_FILES, BASIC_MODEL, constraints, options=options)

assignment = build_jusification_assignments(transitions, ctl, verbose=True)

assignment

Positive Example (Transition) ID: 0
in(0, ())
Positive Example (Transition) ID: 1
in(1, ())
Positive Example (Transition) ID: 2
in(2, ())
Positive Example (Transition) ID: 3
in(3, ())
Positive Example (Transition) ID: 4
in(4, ())
Positive Example (Transition) ID: 5
in(5, ())
Positive Example (Transition) ID: 6
in(6, ())
Positive Example (Transition) ID: 7
in(7, ())
Positive Example (Transition) ID: 8
in(8, ())
Positive Example (Transition) ID: 9
in(9, ())
Positive Example (Transition) ID: 10
in(10, ())
Positive Example (Transition) ID: 11
in(11, ())
Positive Example (Transition) ID: 12
in(12, ())
Positive Example (Transition) ID: 13
in(13, ())
Positive Example (Transition) ID: 14
in(14, ())
Positive Example (Transition) ID: 15
in(15, ())
Positive Example (Transition) ID: 16
in(16, ())


['in(0, ())',
 'in(1, ())',
 'in(2, ())',
 'in(3, ())',
 'in(4, ())',
 'in(5, ())',
 'in(6, ())',
 'in(7, ())',
 'in(8, ())',
 'in(9, ())',
 'in(10, ())',
 'in(11, ())',
 'in(12, ())',
 'in(13, ())',
 'in(14, ())',
 'in(15, ())',
 'in(16, ())']

# Step 3: Counterexample


### State Constraints

In [12]:
state_constraints = []

for state in states:
    parts = [f"obs(holds(SID,{varname},{value},{direction}))"
             for varname, (value, direction) in state.items()]
    constraint = ":- " + ", ".join(parts) + "."
    state_constraints.append(constraint)

state_constraints

[':- obs(holds(SID,growth,none,decrease)), obs(holds(SID,shade,small,decrease)), obs(holds(SID,size,small,decrease)).',
 ':- obs(holds(SID,growth,none,neutral)), obs(holds(SID,shade,small,neutral)), obs(holds(SID,size,small,neutral)).',
 ':- obs(holds(SID,growth,none,increase)), obs(holds(SID,shade,large,increase)), obs(holds(SID,size,large,increase)).',
 ':- obs(holds(SID,growth,positive,increase)), obs(holds(SID,shade,large,increase)), obs(holds(SID,size,large,increase)).',
 ':- obs(holds(SID,growth,positive,increase)), obs(holds(SID,shade,medium,increase)), obs(holds(SID,size,medium,increase)).',
 ':- obs(holds(SID,growth,none,increase)), obs(holds(SID,shade,medium,increase)), obs(holds(SID,size,medium,increase)).',
 ':- obs(holds(SID,growth,none,increase)), obs(holds(SID,shade,small,increase)), obs(holds(SID,size,small,increase)).',
 ':- obs(holds(SID,growth,positive,increase)), obs(holds(SID,shade,small,increase)), obs(holds(SID,size,small,increase)).']

#### Find a counterexample

i.e. a state, that does not cause a violation, but is also not in the positive examples.

In [13]:
def counterexamples_for_model(ASPGARPFILES: list[str], MFILES: list[str], assumed_model, state_constraints, 
) -> list[dict[str, tuple[str, str]]]:
    states: list[dict[str, tuple[str, str]]] = []

    ctl = clingo.Control(["1", "--project", "--warn=none"])
    for path in ASPGARPFILES:
        ctl.load(path)

    for path in MFILES:
        ctl.load(path)

    ctl.add("base", [], "#show obs/1.")
    ctl.add("base", [], ":- violation.")

    for constraint in state_constraints:
        ctl.add("base", [], constraint)

    for model_part in assumed_model:
        ctl.add("base", [], f"{model_part}.")

    ctl.ground([("base", [])])

    with ctl.solve(yield_=True) as handle:
        for model in handle:
            state: dict[str, tuple[str, str]] = {}
            for atom in model.symbols(shown=True):
                if atom.name == "obs" and len(atom.arguments) == 1:
                    varname, value, direction = atom.arguments[0].arguments
                    state[varname.name] = (value.name, direction.name)
            states.append(state)

    return states

In [14]:
counterexample = counterexamples_for_model(ASPGARP_FILES,BASIC_MODEL,assumed_model= [], state_constraints= state_constraints)
counterexample

[{'growth': ('positive', 'increase'),
  'shade': ('small', 'decrease'),
  'size': ('small', 'decrease')}]

# Step 4: Create Minimum Satisfiable Core Assignment


In [15]:
import clingo
from typing import Any, Iterable, Mapping

def make_clingo_control(
    aspgarp_files: Iterable[str],
    meta_files: Iterable[str],
    model_files: Iterable[str],
    constraints: Iterable[str],
    *,
    options: Iterable[str] = [],
    parallel_mode: int = 4,
) -> clingo.Control:
    """Create and ground a clingo Control object."""
    ctl = clingo.Control([
        "0",
        "--project",
        "--warn=none",
        f"--parallel-mode={parallel_mode}",
        *options,
    ])

    for path in [*aspgarp_files, *meta_files, *model_files]:
        ctl.load(path)

    ctl.add("base", [], "\n".join(constraints))
    ctl.add("base", [], "#show abduced/1.")
    ctl.ground([("base", [])])
    return ctl


def build_assignment(
    counterexample,
    ctl: clingo.Control,
    key: int = 0,
    *,
    require_violation: bool = True,
    verbose: bool = False,
) -> list[str]:
    """Solve each counterexample and return assignment strings."""
    violation = clingo.Function("violation")
    assignments: list[str] = []
    term_cache: dict[str, clingo.Symbol] = {}
    
    assumptions =  state_to_assumptions(counterexample)
    print(assumptions)

    if require_violation:
        assumptions.append((violation, True))

    with ctl.solve(yield_=True, assumptions=assumptions) as handle:
        for model in handle:
            shown = sorted(model.symbols(shown=True), key=str)
            rule = "(" + ",".join(map(str, shown)) + ")"
            assignment = f"in({key}, {rule})"
            assignments.append(assignment)

            if verbose:
                print(assignment)

    return assignments

In [16]:
ctl = make_clingo_control(ASPGARP_FILES, META_FILES, BASIC_MODEL, state_constraints, options=["--const", "c = 1"])

assignment = build_assignment(counterexample[0], ctl,key = 0, verbose=True)

assignment

[(Function('obs', [Function('holds', [Function('growth', [], True), Function('positive', [], True), Function('increase', [], True)], True)], True), True), (Function('obs', [Function('holds', [Function('shade', [], True), Function('small', [], True), Function('decrease', [], True)], True)], True), True), (Function('obs', [Function('holds', [Function('size', [], True), Function('small', [], True), Function('decrease', [], True)], True)], True), True)]
in(0, (abduced(influence(1,growth,shade))))
in(0, (abduced(correspondence(growth,size))))
in(0, (abduced(influence(1,growth,size))))
in(0, (abduced(influence(-1,growth,shade))))
in(0, (abduced(proportionality(-1,shade,size))))
in(0, (abduced(correspondence(size,growth))))
in(0, (abduced(correspondence(shade,growth))))
in(0, (abduced(correspondence(growth,shade))))
in(0, (abduced(proportionality(1,shade,growth))))
in(0, (abduced(proportionality(1,growth,shade))))
in(0, (abduced(proportionality(1,growth,size))))
in(0, (abduced(proportionality

['in(0, (abduced(influence(1,growth,shade))))',
 'in(0, (abduced(correspondence(growth,size))))',
 'in(0, (abduced(influence(1,growth,size))))',
 'in(0, (abduced(influence(-1,growth,shade))))',
 'in(0, (abduced(proportionality(-1,shade,size))))',
 'in(0, (abduced(correspondence(size,growth))))',
 'in(0, (abduced(correspondence(shade,growth))))',
 'in(0, (abduced(correspondence(growth,shade))))',
 'in(0, (abduced(proportionality(1,shade,growth))))',
 'in(0, (abduced(proportionality(1,growth,shade))))',
 'in(0, (abduced(proportionality(1,growth,size))))',
 'in(0, (abduced(proportionality(1,size,growth))))',
 'in(0, (abduced(proportionality(-1,growth,shade))))',
 'in(0, (abduced(influence(-1,growth,size))))',
 'in(0, (abduced(proportionality(-1,size,shade))))',
 'in(0, (abduced(proportionality(-1,growth,size))))']

In [17]:
assignments = []

#expand with assignment

assignments.extend(assignment)

In [18]:
import clingo
from pathlib import Path
from typing import Iterable


def extract_unique_abduced_facts(
    hs_file: str,
    *,
    assignment: Iterable[str] | None = None,
    assignment_file: str | None = None,
) -> list[str]:
    """
    Solve hs_file plus optional assignment facts, take the optimal model,
    and return abduced facts with fresh pc/rl ids per hit.
    """
    ctl = clingo.Control(["0", "--project", "--warn=none"])

    if assignment_file is not None:
        ctl.load(assignment_file)
    elif assignment is not None:
        ctl.add("base", [], "\n".join(f"{a}." for a in assignment))

    ctl.load(hs_file)
    ctl.ground([("base", [])])

    def renumber(sym: clingo.Symbol, i: int) -> clingo.Symbol:
        if sym.type != clingo.SymbolType.Function:
            return sym
        args = [renumber(a, i) for a in sym.arguments]
        if sym.name in {"pc", "rl"} and len(args) == 1:
            return clingo.Function(sym.name, [clingo.Number(i)])
        return clingo.Function(sym.name, args, sym.positive)

    hits: list[clingo.Symbol] = []
    with ctl.solve(yield_=True) as handle:
        for model in handle:
            hits = [a for a in model.symbols(shown=True) if a.name == "hit"]

    facts: list[str] = []
    for i, hit in enumerate(hits, start=1):
        arguments = hit.arguments[0].arguments
        for argument in arguments:
            if argument.type == clingo.SymbolType.Function and argument.name in {"pc", "rl"}:
                continue  # skip pc/rl terms
            else:
                facts.append(str(renumber(argument, i)))


    return facts

# Step 5: Hitting Set

In [19]:
new_facts = extract_unique_abduced_facts(f"{project_root}/hs.lp", assignment=assignment)

new_facts

['influence(1,growth,size)']

# Step 6: Rinse and Repeat

In [20]:
import time

hsassignment = list(assignment)
learned_state_constraints = list(state_constraints)
MAX_ITERATIONS = 100
unsat_counterexamples: list[dict[str, str]] = []

ctl = make_clingo_control(
        ASPGARP_FILES,
        META_FILES,
        BASIC_MODEL,
        constraints,
        options=["--const", "c=1"],)

loop_start = time.perf_counter()

completed_iterations = 0

for iteration in range(MAX_ITERATIONS):
    completed_iterations = iteration
    iter_start = time.perf_counter()
    print(f"\n=== Iteration {iteration + 1} ===")

    t0 = time.perf_counter()
    abduced_qsm = extract_unique_abduced_facts("hs.lp", assignment=hsassignment)
    print(f"  [timing] extract_unique_abduced_facts: {time.perf_counter() - t0:.3f}s")
    for fact in abduced_qsm:
        print("  ", fact)

    t0 = time.perf_counter()
    new_counterexample = counterexamples_for_model(
        ASPGARP_FILES,
        BASIC_MODEL,
        abduced_qsm,
        learned_state_constraints,        
    )
    print(f"  [timing] counterexample: {time.perf_counter() - t0:.3f}s")

    if not new_counterexample:
        print("No counterexample found, stopping.")
        break

    t0 = time.perf_counter()

    print(f"  [timing] make_clingo_control: {time.perf_counter() - t0:.3f}s")

    t0 = time.perf_counter()
    new_assignment = build_assignment(
        counterexample[0],
        ctl,
        key = iteration + 1,
        verbose=True,
    )
    print(f"  [timing] build_assignments: {time.perf_counter() - t0:.3f}s")

    if not new_assignment:

        print("No assignment found for this counterexample.")
        break
        print("Adding it as a state constraint and continuing.")
        unsat_counterexamples.append(new_neg_example)
        atoms = [
            part.strip()
            for part in new_neg_example["program"].split(".")
            if part.strip()
        ]
        learned_constraint = ":- " + ", ".join(atoms) + "."
        learned_state_constraints.append(learned_constraint)
        print("Learned state constraint:")
        print("  ", learned_constraint)
        print(f"  [timing] iteration {iteration + 1} total: {time.perf_counter() - iter_start:.3f}s")
        
        continue

    print("New assignment from counterexample:")
    # for a in new_assignment:
    #     print("  ", a)
    hsassignment.extend(new_assignment)

    #write assignemnt to file:
    with open(f"{OUTPUT}/assignment.lp", "w") as f:
        for a in hsassignment:
            f.write(f"{a}.\n")

    print(f"  [timing] iteration {iteration + 1} total: {time.perf_counter() - iter_start:.3f}s")

print(f"\n[timing] total loop: {time.perf_counter() - loop_start:.3f}s")


=== Iteration 1 ===
  [timing] extract_unique_abduced_facts: 0.001s
   influence(1,growth,size)
  [timing] counterexample: 0.005s
  [timing] make_clingo_control: 0.000s
[(Function('obs', [Function('holds', [Function('growth', [], True), Function('positive', [], True), Function('increase', [], True)], True)], True), True), (Function('obs', [Function('holds', [Function('shade', [], True), Function('small', [], True), Function('decrease', [], True)], True)], True), True), (Function('obs', [Function('holds', [Function('size', [], True), Function('small', [], True), Function('decrease', [], True)], True)], True), True)]
in(1, (abduced(influence(1,growth,shade))))
in(1, (abduced(influence(1,growth,size))))
in(1, (abduced(proportionality(1,shade,growth))))
in(1, (abduced(proportionality(1,growth,size))))
in(1, (abduced(proportionality(1,size,growth))))
in(1, (abduced(proportionality(1,growth,shade))))
  [timing] build_assignments: 0.001s
New assignment from counterexample:
  [timing] iterati